<a href="https://colab.research.google.com/github/LucasCAraujo21/VendingMachine/blob/main/Aula08_Pratica3_VendingMachine_Completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Aula 08 — Prática 3: Autômatos Finitos na Prática
### Compiladores — FMU | Prof. Hercules Ramos | 2026

Notebook com **todos os exemplos** da Aula 08 e o **ponto de partida do projeto**.
O passo a passo completo, com desenhos e explicações, está no material da aula.

> 💾 Antes de começar: **Arquivo → Salvar uma cópia no Drive** e trabalhe na sua cópia.
>
> ▶️ Execute as células **em ordem**: as Células 3 a 9 usam a classe `AFD` e os autômatos criados antes.

**A regra de ouro desta prática:** todo autômato é um dicionário
`delta = {(estado_atual, símbolo): próximo_estado}`. Se a chave não existe, o autômato rejeita.

In [1]:
# ⚙️ CÉLULA 0 — Preparação do ambiente
# Tudo nesta aula usa só a biblioteca padrão do Python.
# Funciona no Google Colab e no VS Code (Jupyter) sem instalar nada.
import re

print("✅ Ambiente pronto! Vamos construir autômatos.")

✅ Ambiente pronto! Vamos construir autômatos.


---
## 💡 Exemplo 1 — O interruptor de luz
Dois estados (`APAGADA`, `ACESA`), um símbolo (`clique`). Aceita quando a luz termina **acesa**.

🔧 Experimente: troque a lista `eventos` por 2 ou 4 cliques.

In [2]:
# 💡 CÉLULA 1 — Exemplo 1: o interruptor de luz (o menor autômato do mundo)
# Estados:  APAGADA, ACESA
# Alfabeto: {"clique"}
# δ (delta): um dicionário  (estado_atual, símbolo) -> próximo_estado

delta_lampada = {
    ("APAGADA", "clique"): "ACESA",
    ("ACESA", "clique"): "APAGADA",
}

estado = "APAGADA"                      # estado inicial
eventos = ["clique", "clique", "clique"]

for evento in eventos:
    novo = delta_lampada[(estado, evento)]
    print(f"{estado:8} --{evento}--> {novo}")
    estado = novo

print("Estado final:", estado)
print("Aceita? (a luz terminou acesa)", estado == "ACESA")

APAGADA  --clique--> ACESA
ACESA    --clique--> APAGADA
APAGADA  --clique--> ACESA
Estado final: ACESA
Aceita? (a luz terminou acesa) True


---
## 🚗 Exemplo 2 — Placa Mercosul (`ABC1D23`)
```
 ──► (q0) ─L─► (q1) ─L─► (q2) ─L─► (q3) ─D─► (q4) ─L─► (q5) ─D─► (q6) ─D─► ((q7))
```
Em vez de uma seta por letra, usamos **classes**: `L` = letra, `D` = dígito.

In [3]:
# 🚗 CÉLULA 2 — Exemplo 2: validando uma placa Mercosul (ABC1D23)
# Padrão: Letra Letra Letra Dígito Letra Dígito Dígito
#
# Em vez de criar uma transição para cada letra (26!) e cada dígito (10!),
# agrupamos os caracteres em CLASSES:  "L" = letra,  "D" = dígito.

def classe_placa(c):
    if "A" <= c <= "Z":
        return "L"
    if "0" <= c <= "9":
        return "D"
    return "outro"


delta_placa = {
    ("q0", "L"): "q1",
    ("q1", "L"): "q2",
    ("q2", "L"): "q3",
    ("q3", "D"): "q4",
    ("q4", "L"): "q5",
    ("q5", "D"): "q6",
    ("q6", "D"): "q7",
}
FINAIS_PLACA = {"q7"}


def aceita_placa(texto):
    estado = "q0"
    for c in texto:
        chave = (estado, classe_placa(c))
        if chave not in delta_placa:     # não existe seta: caiu no "estado morto"
            return False
        estado = delta_placa[chave]
    return estado in FINAIS_PLACA        # terminou num estado de aceitação?


for placa in ["ABC1D23", "BRA2E19", "ABC1234", "AB1C234", "abc1d23", "ABC1D2"]:
    print(f"{placa:8} -> {'✅ válida' if aceita_placa(placa) else '❌ inválida'}")

ABC1D23  -> ✅ válida
BRA2E19  -> ✅ válida
ABC1234  -> ❌ inválida
AB1C234  -> ❌ inválida
abc1d23  -> ❌ inválida
ABC1D2   -> ❌ inválida


---
## 🧰 Exemplo 3 — Uma classe `AFD` reutilizável
O "motor" é sempre o mesmo: só muda a tabela δ. A classe guarda **M = (Q, Σ, δ, q0, F)** e ganha:
- `rodar(palavra)`: passo a passo da leitura;
- `tabela()`: a tabela de transição, igual à da apostila.

In [4]:
# 🧰 CÉLULA 3 — Exemplo 3: uma classe AFD reutilizável (com rastreamento passo a passo)
# A ideia: escrever o "motor" UMA vez e só trocar a tabela δ para cada problema.
# Um AFD é a 5-upla  M = (Q, Σ, δ, q0, F).

class AFD:
    def __init__(self, nome, delta, inicial, finais, classificar):
        self.nome = nome
        self.delta = delta                  # δ: {(estado, classe): próximo}
        self.inicial = inicial              # q0
        self.finais = set(finais)           # F
        self.classificar = classificar      # função: caractere -> classe (Σ)
        self.estados = sorted({e for e, _ in delta} | set(delta.values()))           # Q
        self.alfabeto = sorted({s for _, s in delta})                               # Σ

    def rodar(self, palavra, mostrar=True):
        estado = self.inicial
        if mostrar:
            print(f"🔎 {self.nome} lendo {palavra!r}")
            print(f"   {'passo':5} {'lido':5} {'classe':7} {'de':6} {'para':6}")
        for passo, c in enumerate(palavra, start=1):
            classe = self.classificar(c)
            proximo = self.delta.get((estado, classe))
            if mostrar:
                print(f"   {passo:<5} {c!r:5} {classe:7} {estado:6} {proximo or '💀 morto'}")
            if proximo is None:
                if mostrar:
                    print(f"   ❌ REJEITADA: não há transição de {estado} com {classe!r}\n")
                return False
            estado = proximo
        aceita = estado in self.finais
        if mostrar:
            if aceita:
                print(f"   ✅ ACEITA: terminou em {estado}, que é de aceitação\n")
            else:
                print(f"   ❌ REJEITADA: terminou em {estado}, que NÃO é de aceitação\n")
        return aceita

    def tabela(self):
        """Imprime a tabela de transição: linhas = estados, colunas = classes."""
        print(f"📋 Tabela de transição — {self.nome}")
        print("   " + f"{'estado':10}" + "".join(f"{s:>7}" for s in self.alfabeto))
        for q in self.estados:
            marca = ("→" if q == self.inicial else " ") + ("*" if q in self.finais else " ")
            linha = "".join(f"{self.delta.get((q, s), '—'):>7}" for s in self.alfabeto)
            print(f"{marca} {q:10}{linha}")
        print("   (→ = inicial, * = aceitação, — = vai para o estado morto)\n")


placa = AFD("Placa Mercosul", delta_placa, "q0", {"q7"}, classe_placa)
placa.tabela()
placa.rodar("BRA2E19")
placa.rodar("BRA2019")

📋 Tabela de transição — Placa Mercosul
   estado          D      L
→  q0              —     q1
   q1              —     q2
   q2              —     q3
   q3             q4      —
   q4              —     q5
   q5             q6      —
   q6             q7      —
 * q7              —      —
   (→ = inicial, * = aceitação, — = vai para o estado morto)

🔎 Placa Mercosul lendo 'BRA2E19'
   passo lido  classe  de     para  
   1     'B'   L       q0     q1
   2     'R'   L       q1     q2
   3     'A'   L       q2     q3
   4     '2'   D       q3     q4
   5     'E'   L       q4     q5
   6     '1'   D       q5     q6
   7     '9'   D       q6     q7
   ✅ ACEITA: terminou em q7, que é de aceitação

🔎 Placa Mercosul lendo 'BRA2019'
   passo lido  classe  de     para  
   1     'B'   L       q0     q1
   2     'R'   L       q1     q2
   3     'A'   L       q2     q3
   4     '2'   D       q3     q4
   5     '0'   D       q4     💀 morto
   ❌ REJEITADA: não há transição de q4 com 'D'



False

---
## ⏰ Exemplo 4 — Horário HH:MM (00:00 a 23:59)
Se a hora começa com `2`, o segundo dígito vai só até `3`. Para **lembrar** disso, o autômato usa dois estados diferentes: `H1a` e `H1b`.

> **Estado = memória.**

In [5]:
# ⏰ CÉLULA 4 — Exemplo 4: horário HH:MM (de 00:00 até 23:59)
# O truque: a HORA não pode ser 24, 25... Se começar com 2, o próximo dígito vai só até 3.
# Por isso as classes de dígitos são mais "finas" que no exemplo da placa.

def classe_hora(c):
    if c in "01":
        return "0-1"
    if c == "2":
        return "2"
    if c == "3":
        return "3"
    if c in "45":
        return "4-5"
    if c in "6789":
        return "6-9"
    if c == ":":
        return ":"
    return "outro"


QUALQUER_DIGITO = ["0-1", "2", "3", "4-5", "6-9"]

delta_hora = {("H0", "0-1"): "H1a", ("H0", "2"): "H1b"}      # 1º dígito da hora
for d in QUALQUER_DIGITO:
    delta_hora[("H1a", d)] = "H2"                            # 0x ou 1x: qualquer dígito
for d in ["0-1", "2", "3"]:
    delta_hora[("H1b", d)] = "H2"                            # 2x: só de 0 a 3
delta_hora[("H2", ":")] = "M0"                               # os dois-pontos
for d in ["0-1", "2", "3", "4-5"]:
    delta_hora[("M0", d)] = "M1"                             # 1º dígito do minuto: 0 a 5
for d in QUALQUER_DIGITO:
    delta_hora[("M1", d)] = "M2"                             # 2º dígito do minuto: qualquer

hora = AFD("Horário HH:MM", delta_hora, "H0", {"M2"}, classe_hora)
hora.tabela()
hora.rodar("23:59")
hora.rodar("24:00")
for h in ["00:00", "09:30", "19:45", "7:30", "12:60", "12h30"]:
    print(f"{h:6} -> {'✅' if hora.rodar(h, mostrar=False) else '❌'}")

📋 Tabela de transição — Horário HH:MM
   estado        0-1      2      3    4-5    6-9      :
→  H0            H1a    H1b      —      —      —      —
   H1a            H2     H2     H2     H2     H2      —
   H1b            H2     H2     H2      —      —      —
   H2              —      —      —      —      —     M0
   M0             M1     M1     M1     M1      —      —
   M1             M2     M2     M2     M2     M2      —
 * M2              —      —      —      —      —      —
   (→ = inicial, * = aceitação, — = vai para o estado morto)

🔎 Horário HH:MM lendo '23:59'
   passo lido  classe  de     para  
   1     '2'   2       H0     H1b
   2     '3'   3       H1b    H2
   3     ':'   :       H2     M0
   4     '5'   4-5     M0     M1
   5     '9'   6-9     M1     M2
   ✅ ACEITA: terminou em M2, que é de aceitação

🔎 Horário HH:MM lendo '24:00'
   passo lido  classe  de     para  
   1     '2'   2       H0     H1b
   2     '4'   4-5     H1b    💀 morto
   ❌ REJEITADA: não há transiçã

---
## 🔁 Exemplo 5 — AFD × Expressão Regular
Se o AFD desenhado à mão e a RegEx concordam em todos os testes, temos boa confiança de que o desenho está certo.

⚠️ Use `re.fullmatch` (o texto **inteiro** precisa casar), e não `re.search`.

In [6]:
# 🔁 CÉLULA 5 — Exemplo 5: AFD × Expressão Regular (eles têm que concordar!)
# Toda RegEx tem um AFD equivalente (Thompson + subconjuntos, Aula 07).
# Aqui fazemos o caminho inverso: desenhamos o AFD à mão e usamos a RegEx para conferir.

REGEX_PLACA = r"[A-Z]{3}[0-9][A-Z][0-9]{2}"
REGEX_HORA = r"([01][0-9]|2[0-3]):[0-5][0-9]"


def comparar(afd, regex, palavras):
    divergencias = 0
    print(f"{'palavra':10} {'AFD':5} {'RegEx':5}")
    for p in palavras:
        r_afd = afd.rodar(p, mostrar=False)
        r_re = re.fullmatch(regex, p) is not None
        ok = "✅" if r_afd == r_re else "⚠️ DIVERGEM"
        divergencias += r_afd != r_re
        print(f"{p:10} {str(r_afd):5} {str(r_re):5} {ok}")
    print(f"→ divergências: {divergencias}\n")


comparar(placa, REGEX_PLACA, ["ABC1D23", "XYZ9A00", "ABC1234", "ABCD123", "", "AB1D23"])
comparar(hora, REGEX_HORA, ["00:00", "23:59", "20:15", "24:00", "29:10", "10:7", "1:00"])

palavra    AFD   RegEx
ABC1D23    True  True  ✅
XYZ9A00    True  True  ✅
ABC1234    False False ✅
ABCD123    False False ✅
           False False ✅
AB1D23     False False ✅
→ divergências: 0

palavra    AFD   RegEx
00:00      True  True  ✅
23:59      True  True  ✅
20:15      True  True  ✅
24:00      False False ✅
29:10      False False ✅
10:7       False False ✅
1:00       False False ✅
→ divergências: 0



---
## 📦 Exemplo 6 — O ciclo de vida de um pedido
Aqui os símbolos são **eventos** (`pagar`, `separar`, `enviar`...). Depois de `ENVIADO`, não existe a seta `cancelar`.

In [7]:
# 📦 CÉLULA 6 — Exemplo 6: o ciclo de vida de um pedido (autômato fora do compilador)
# Os "símbolos" aqui não são letras: são EVENTOS do sistema.
# É o mesmo modelo usado em e-commerce, apps de delivery e fluxos de pagamento.

delta_pedido = {
    ("CRIADO", "pagar"): "PAGO",
    ("CRIADO", "cancelar"): "CANCELADO",
    ("PAGO", "separar"): "SEPARADO",
    ("PAGO", "cancelar"): "CANCELADO",
    ("SEPARADO", "enviar"): "ENVIADO",
    ("SEPARADO", "cancelar"): "CANCELADO",
    ("ENVIADO", "entregar"): "ENTREGUE",
}
FINAIS_PEDIDO = {"ENTREGUE", "CANCELADO"}      # o pedido "terminou"


def processar_pedido(eventos):
    estado = "CRIADO"
    print(f"🛒 eventos: {eventos}")
    for i, ev in enumerate(eventos, start=1):
        proximo = delta_pedido.get((estado, ev))
        if proximo is None:
            permitidos = [e for (q, e) in delta_pedido if q == estado]
            print(f"   ❌ evento {i} '{ev}' não é permitido em {estado}. Permitidos: {permitidos}\n")
            return False
        print(f"   {estado:9} --{ev}--> {proximo}")
        estado = proximo
    if estado in FINAIS_PEDIDO:
        print(f"   ✅ pedido finalizado em {estado}\n")
        return True
    print(f"   ⏳ pedido ainda em andamento ({estado})\n")
    return False


processar_pedido(["pagar", "separar", "enviar", "entregar"])
processar_pedido(["pagar", "cancelar"])
processar_pedido(["pagar", "enviar"])              # pulou a separação!
processar_pedido(["pagar", "separar", "enviar", "cancelar"])  # já saiu para entrega

🛒 eventos: ['pagar', 'separar', 'enviar', 'entregar']
   CRIADO    --pagar--> PAGO
   PAGO      --separar--> SEPARADO
   SEPARADO  --enviar--> ENVIADO
   ENVIADO   --entregar--> ENTREGUE
   ✅ pedido finalizado em ENTREGUE

🛒 eventos: ['pagar', 'cancelar']
   CRIADO    --pagar--> PAGO
   PAGO      --cancelar--> CANCELADO
   ✅ pedido finalizado em CANCELADO

🛒 eventos: ['pagar', 'enviar']
   CRIADO    --pagar--> PAGO
   ❌ evento 2 'enviar' não é permitido em PAGO. Permitidos: ['separar', 'cancelar']

🛒 eventos: ['pagar', 'separar', 'enviar', 'cancelar']
   CRIADO    --pagar--> PAGO
   PAGO      --separar--> SEPARADO
   SEPARADO  --enviar--> ENVIADO
   ❌ evento 4 'cancelar' não é permitido em ENVIADO. Permitidos: ['entregar']



False

---
## 🛰️ Exemplo 7 — Mini-scanner da linguagem "Calc"
Cada `while` é um estado do autômato que continua girando enquanto o caractere pertence ao token (**Maximal Munch**).

🤔 Teste: o que acontece com `25.` e com `.5`? Explique pelo código.

In [8]:
# 🛰️ CÉLULA 7 — Exemplo 7: um mini-scanner feito à mão (linguagem "Calc")
# Cada laço "while" abaixo é um ESTADO do autômato que fica girando enquanto
# o caractere pertence ao token. Isso é o Maximal Munch: ler o MAIOR lexema possível.

PALAVRAS_RESERVADAS = {"seja", "mostre", "se"}
SIMPLES = {"+": "MAIS", "-": "MENOS", "*": "VEZES", "/": "DIVIDE", "(": "ABRE_PAR", ")": "FECHA_PAR"}


def escanear(fonte):
    tokens, erros = [], []
    i, linha, coluna = 0, 1, 1
    while i < len(fonte):
        c = fonte[i]
        inicio, col_inicio = i, coluna

        if c == "\n":                                   # quebra de linha: só atualiza a posição
            i, linha, coluna = i + 1, linha + 1, 1
            continue
        if c in " \t":                                  # espaço: ignorado
            i, coluna = i + 1, coluna + 1
            continue
        if c == "#":                                    # comentário: ignora até o fim da linha
            while i < len(fonte) and fonte[i] != "\n":
                i += 1
            coluna += i - inicio
            continue

        if c.isalpha() or c == "_":                     # estado ID: letras, dígitos e _
            while i < len(fonte) and (fonte[i].isalnum() or fonte[i] == "_"):
                i += 1
            lexema = fonte[inicio:i]
            tipo = "RESERVADA" if lexema in PALAVRAS_RESERVADAS else "ID"
        elif c.isdigit():                               # estado NUM: dígitos e, talvez, .dígitos
            while i < len(fonte) and fonte[i].isdigit():
                i += 1
            tipo = "NUM_INT"
            if i + 1 < len(fonte) and fonte[i] == "." and fonte[i + 1].isdigit():
                i += 1
                while i < len(fonte) and fonte[i].isdigit():
                    i += 1
                tipo = "NUM_REAL"
            lexema = fonte[inicio:i]
        elif c == "=":                                  # "=" ou "==" ? Maximal Munch decide
            if i + 1 < len(fonte) and fonte[i + 1] == "=":
                i, tipo = i + 2, "IGUAL"
            else:
                i, tipo = i + 1, "ATRIB"
            lexema = fonte[inicio:i]
        elif c in SIMPLES:
            i, tipo, lexema = i + 1, SIMPLES[c], c
        else:                                           # nenhum estado aceita: erro léxico
            dica = "use ponto como separador decimal" if c == "," else "caractere fora da linguagem"
            erros.append(f"Linha {linha}, coluna {coluna}: caractere inesperado {c!r} ({dica})")
            i, coluna = i + 1, coluna + 1
            continue

        tokens.append((tipo, lexema, linha, col_inicio))
        coluna += i - inicio
    return tokens, erros


programa = """# calcula o total do pedido
seja preco = 25.90
seja frete = 10
seja total = preco + frete * 2
se total == 45.90
mostre total
seja taxa = 2,5 @"""

tokens, erros = escanear(programa)
print(f"{'TIPO':10} {'LEXEMA':8} {'LIN':>3} {'COL':>3}")
for tipo, lexema, lin, col in tokens:
    print(f"{tipo:10} {lexema:8} {lin:>3} {col:>3}")
print()
for e in erros:
    print("⚠️", e)

TIPO       LEXEMA   LIN COL
RESERVADA  seja       2   1
ID         preco      2   6
ATRIB      =          2  12
NUM_REAL   25.90      2  14
RESERVADA  seja       3   1
ID         frete      3   6
ATRIB      =          3  12
NUM_INT    10         3  14
RESERVADA  seja       4   1
ID         total      4   6
ATRIB      =          4  12
ID         preco      4  14
MAIS       +          4  20
ID         frete      4  22
VEZES      *          4  28
NUM_INT    2          4  30
RESERVADA  se         5   1
ID         total      5   4
IGUAL      ==         5  10
NUM_REAL   45.90      5  13
RESERVADA  mostre     6   1
ID         total      6   8
RESERVADA  seja       7   1
ID         taxa       7   6
ATRIB      =          7  11
NUM_INT    2          7  13
NUM_INT    5          7  15

⚠️ Linha 7, coluna 14: caractere inesperado ',' (use ponto como separador decimal)
⚠️ Linha 7, coluna 17: caractere inesperado '@' (caractere fora da linguagem)


---
## 🧪 Laboratório interativo
Escolha o autômato, digite uma palavra e clique em **Rodar**.
> No VS Code, se faltar o pacote: `pip install ipywidgets`.

In [9]:
# 🧪 CÉLULA 8 — Laboratório interativo: teste os autômatos da aula
# Digite uma palavra, escolha o autômato e veja o passo a passo.
import ipywidgets as widgets
from IPython.display import display

AUTOMATOS = {"Placa Mercosul": placa, "Horário HH:MM": hora}

escolha = widgets.Dropdown(options=list(AUTOMATOS), description="Autômato:")
entrada = widgets.Text(value="BRA2E19", description="Palavra:")
botao = widgets.Button(description="Rodar", button_style="primary")
saida = widgets.Output()


def ao_clicar(_):
    saida.clear_output()
    with saida:
        AUTOMATOS[escolha.value].rodar(entrada.value)


botao.on_click(ao_clicar)
display(widgets.HBox([escolha, entrada, botao]), saida)

Output()

---
# 🎯 PROJETO — Validador & Scanner do seu domínio
O enunciado completo (requisitos R1 a R7, bônus, rubrica e **sugestões de temas**) está no material da aula.

**Entrega:** grupo de 4 a 6 alunos · repositório **público** no GitHub · README com o **nome de todos os integrantes no começo** · apenas um membro entrega.

A célula abaixo é o **ponto de partida**: troque o exemplo (números inteiros) pelo seu primeiro validador.

> 💡 Crie as suas alterações em **novas células abaixo**, sem apagar o código da aula. Para o scanner, use a função `escanear` do Exemplo 7 como modelo.

In [10]:
# 🎯 CÉLULA 9 — PROJETO: ponto de partida (substitua o exemplo pelo SEU tema)
# Reaproveite a classe AFD (Célula 3) e a função escanear (Célula 7) como modelo.

# ---------- R1: um AFD do seu domínio ----------
def minha_classe(c):
    # TODO: transforme o caractere em uma classe (ex.: "L", "D", "-", "outro")
    if c.isdigit():
        return "D"
    return "outro"


meu_delta = {
    ("q0", "D"): "q1",
    ("q1", "D"): "q1",
    # TODO: desenhe o autômato no caderno ANTES e só depois escreva as setas aqui
}

meu_afd = AFD("TODO: nome do validador", meu_delta, "q0", {"q1"}, minha_classe)
meu_afd.tabela()

# ---------- R3: bateria de testes (AFD × RegEx) ----------
MINHA_REGEX = r"[0-9]+"        # TODO: a RegEx equivalente ao seu AFD
casos = {
    "123": True,               # TODO: mínimo de 5 aceitas e 5 rejeitadas por AFD
    "12a": False,
}
for palavra, esperado in casos.items():
    r_afd = meu_afd.rodar(palavra, mostrar=False)
    r_re = re.fullmatch(MINHA_REGEX, palavra) is not None
    ok = "✅" if r_afd == r_re == esperado else "❌"
    print(f"{palavra:12} esperado={esperado!s:5} AFD={r_afd!s:5} RegEx={r_re!s:5} {ok}")

📋 Tabela de transição — TODO: nome do validador
   estado          D
→  q0             q1
 * q1             q1
   (→ = inicial, * = aceitação, — = vai para o estado morto)

123          esperado=True  AFD=True  RegEx=True  ✅
12a          esperado=False AFD=False RegEx=False ✅


---
### 📚 Referências
- CARMO, Robson. *Compiladores*. FMU, 2026 — Unidades 5 e 6.
- AHO, A. et al. *Compiladores: Princípios, Técnicas e Ferramentas*. 2. ed. Pearson, 2008 — Cap. 3.
- MENEZES, P. B. *Linguagens Formais e Autômatos*. 6. ed. Bookman, 2011.
- Documentação do módulo `re`: https://docs.python.org/3/library/re.html
- Documentação do ipywidgets: https://ipywidgets.readthedocs.io

---
# ☕ PROJETO — Autômatos e Mini-scanner de uma Vending Machine

In [11]:
# R1 — AFD 1: código do produto A1 até F9 (5 estados)
def classe_codigo_produto(c):
    if "A" <= c <= "C":
        return "A-C"
    if "D" <= c <= "F":
        return "D-F"
    if "1" <= c <= "9":
        return "1-9"
    return "outro"


delta_codigo_produto = {
    ("q0", "A-C"): "q_ABC",
    ("q0", "D-F"): "q_DEF",
    ("q_ABC", "1-9"): "qf_ABC",
    ("q_DEF", "1-9"): "qf_DEF",
}
REGEX_CODIGO_PRODUTO = r"[A-F][1-9]"
afd_codigo_produto = AFD(
    "Código do produto A1-F9",
    delta_codigo_produto,
    "q0",
    {"qf_ABC", "qf_DEF"},
    classe_codigo_produto,
)
afd_codigo_produto.tabela()

📋 Tabela de transição — Código do produto A1-F9
   estado        1-9    A-C    D-F
→  q0              —  q_ABC  q_DEF
   q_ABC      qf_ABC      —      —
   q_DEF      qf_DEF      —      —
 * qf_ABC          —      —      —
 * qf_DEF          —      —      —
   (→ = inicial, * = aceitação, — = vai para o estado morto)



In [12]:
# R1 — AFD 2: denominações aceitas (estados finos para partes do valor)
def classe_moeda(c):
    if c in {"0", "1", "2", "5", "."}:
        return c
    return "outro"


delta_moeda = {
    ("m0", "0"): "mZ",             # começa com zero
    ("m0", "1"): "m1",             # começa com um
    ("mZ", "0"): "mZ",             # zeros à esquerda
    ("mZ", "1"): "m1",             # aceita 01.00, 001.00 etc.
    ("mZ", "."): "mZdot",
    ("m1", "."): "m1dot",
    ("mZdot", "2"): "m25",
    ("mZdot", "5"): "m50",
    ("m25", "5"): "mf25",
    ("m50", "0"): "mf50",
    ("m1dot", "0"): "m100",
    ("m100", "0"): "mf100",
}
REGEX_MOEDA = r"0*(?:0\.(?:25|50)|1\.00)"
afd_moeda = AFD(
    "Moeda aceita (0.25, 0.50, 1.00)",
    delta_moeda,
    "m0",
    {"mf25", "mf50", "mf100"},
    classe_moeda,
)
afd_moeda.tabela()

📋 Tabela de transição — Moeda aceita (0.25, 0.50, 1.00)
   estado          .      0      1      2      5
→  m0              —     mZ     m1      —      —
   m1          m1dot      —      —      —      —
   m100            —  mf100      —      —      —
   m1dot           —   m100      —      —      —
   m25             —      —      —      —   mf25
   m50             —   mf50      —      —      —
   mZ          mZdot     mZ     m1      —      —
   mZdot           —      —      —    m25    m50
 * mf100           —      —      —      —      —
 * mf25            —      —      —      —      —
 * mf50            —      —      —      —      —
   (→ = inicial, * = aceitação, — = vai para o estado morto)



## R2 — Rastreamento passo a passo

Para cada AFD, os testes mostram uma palavra aceita e uma rejeitada. Uma rejeição pode ocorrer porque o autômato **morre no meio** (não existe transição) ou porque a palavra termina em um estado que não pertence ao conjunto final `F`.

In [13]:
# R2 — Rastreamento dos dois validadores
print("=" * 65, "\nRastreamento: código de produto")
afd_codigo_produto.rodar("B3")     # aceita: termina em qf_ABC
afd_codigo_produto.rodar("A0")     # rejeita: morre ao ler 0, vindo de q_ABC

Rastreamento: código de produto
🔎 Código do produto A1-F9 lendo 'B3'
   passo lido  classe  de     para  
   1     'B'   A-C     q0     q_ABC
   2     '3'   1-9     q_ABC  qf_ABC
   ✅ ACEITA: terminou em qf_ABC, que é de aceitação

🔎 Código do produto A1-F9 lendo 'A0'
   passo lido  classe  de     para  
   1     'A'   A-C     q0     q_ABC
   2     '0'   outro   q_ABC  💀 morto
   ❌ REJEITADA: não há transição de q_ABC com 'outro'



False

In [14]:
print("=" * 65, "\nRastreamento: moeda")
afd_moeda.rodar("0.25")            # aceita: termina em mf25
afd_moeda.rodar("0.20")            # rejeita: morre ao ler 0 após mZdot
afd_codigo_produto.rodar("A")      # exemplo extra: termina em q_ABC, que não é final

Rastreamento: moeda
🔎 Moeda aceita (0.25, 0.50, 1.00) lendo '0.25'
   passo lido  classe  de     para  
   1     '0'   0       m0     mZ
   2     '.'   .       mZ     mZdot
   3     '2'   2       mZdot  m25
   4     '5'   5       m25    mf25
   ✅ ACEITA: terminou em mf25, que é de aceitação

🔎 Moeda aceita (0.25, 0.50, 1.00) lendo '0.20'
   passo lido  classe  de     para  
   1     '0'   0       m0     mZ
   2     '.'   .       mZ     mZdot
   3     '2'   2       mZdot  m25
   4     '0'   0       m25    💀 morto
   ❌ REJEITADA: não há transição de m25 com '0'

🔎 Código do produto A1-F9 lendo 'A'
   passo lido  classe  de     para  
   1     'A'   A-C     q0     q_ABC
   ❌ REJEITADA: terminou em q_ABC, que NÃO é de aceitação



False

## R3 — Bateria AFD × RegEx

Em cada conjunto há pelo menos cinco entradas aceitas e cinco rejeitadas, incluindo a palavra vazia. A comparação usa `re.fullmatch`, porque o texto inteiro precisa corresponder ao padrão. A meta é **zero divergências** entre o AFD e a expressão regular.

In [15]:
# R3 — Testes do código do produto: 5+ aceitas e 5+ rejeitadas
casos_codigo_produto = [
    "A1", "A9", "C4", "D5", "F9", "E1",   # aceitas
    "", "A0", "G1", "A10", "a1", "F0",   # rejeitadas
]
comparar(afd_codigo_produto, REGEX_CODIGO_PRODUTO, casos_codigo_produto)

# R3 — Testes da moeda: representações das 3 denominações permitidas
casos_moeda = [
    "0.25", "00.25", "000.25", "0.50", "00.50", "1.00", "01.00", "001.00",  # aceitas
    "", "0.00", "0.20", "0.75", "1.50", "2.00", "0,25",                         # rejeitadas
]
comparar(afd_moeda, REGEX_MOEDA, casos_moeda)

palavra    AFD   RegEx
A1         True  True  ✅
A9         True  True  ✅
C4         True  True  ✅
D5         True  True  ✅
F9         True  True  ✅
E1         True  True  ✅
           False False ✅
A0         False False ✅
G1         False False ✅
A10        False False ✅
a1         False False ✅
F0         False False ✅
→ divergências: 0

palavra    AFD   RegEx
0.25       True  True  ✅
00.25      True  True  ✅
000.25     True  True  ✅
0.50       True  True  ✅
00.50      True  True  ✅
1.00       True  True  ✅
01.00      True  True  ✅
001.00     True  True  ✅
           False False ✅
0.00       False False ✅
0.20       False False ✅
0.75       False False ✅
1.50       False False ✅
2.00       False False ✅
0,25       False False ✅
→ divergências: 0



## R4 — Máquina de estados do negócio: ciclo de uma venda

A máquina acompanha o crédito em intervalos de R$ 0,25 até o limite demonstrativo de R$ 1,50. O produto `B3` custa R$ 1,25. Com crédito suficiente, a máquina passa para `LIBERANDO`; depois do evento `TROCO`, encerra em `TROCO` e calcula o valor a devolver. O limite de R$ 1,50 deixa o exemplo finito e torna explícito quais eventos são permitidos em cada estado.

```text
C0 --0.25--> C025 --0.25--> C050 --0.25--> C075 --0.25--> C100 --0.25--> C125 --0.25--> C150
 |             |              |              |              |              |              |
 +--0.50-----> C050           +--0.50-------> C100          +--0.50-------> C150          +--ESCOLHER B3--> LIBERANDO
 +--1.00-----> C100           +--1.00-------> C150                         C125 --ESCOLHER B3--> LIBERANDO
                                                                            LIBERANDO --TROCO--> TROCO
```

Existem transições de R$ 0,50 e R$ 1,00 quando o novo crédito não ultrapassa R$ 1,50. Ao recusar um evento, a função informa os eventos permitidos no estado atual.

In [16]:
from decimal import Decimal

ESTADO_CREDITO = {
    "C0": Decimal("0.00"), "C025": Decimal("0.25"),
    "C050": Decimal("0.50"), "C075": Decimal("0.75"),
    "C100": Decimal("1.00"), "C125": Decimal("1.25"),
    "C150": Decimal("1.50"),
}
PRECO_B3 = Decimal("1.25")

# Eventos em letras maiúsculas, consistentes com o mini-scanner da próxima seção.
delta_vending = {
    ("C0", "INSERIR 0.25"): "C025",
    ("C0", "INSERIR 0.50"): "C050",
    ("C0", "INSERIR 1.00"): "C100",
    ("C025", "INSERIR 0.25"): "C050",
    ("C025", "INSERIR 0.50"): "C075",
    ("C025", "INSERIR 1.00"): "C125",
    ("C050", "INSERIR 0.25"): "C075",
    ("C050", "INSERIR 0.50"): "C100",
    ("C050", "INSERIR 1.00"): "C150",
    ("C075", "INSERIR 0.25"): "C100",
    ("C075", "INSERIR 0.50"): "C125",
    ("C100", "INSERIR 0.25"): "C125",
    ("C100", "INSERIR 0.50"): "C150",
    ("C125", "INSERIR 0.25"): "C150",
    ("C125", "ESCOLHER B3"): "LIBERANDO",
    ("C150", "ESCOLHER B3"): "LIBERANDO",
    ("LIBERANDO", "TROCO"): "TROCO",
}
ESTADOS_FINAIS_VENDING = {"TROCO"}

print("Tabela de transições da vending machine:")
print(f"Total de estados: {len(set(q for q, _ in delta_vending) | set(delta_vending.values()))}")
print(f"Total de transições: {len(delta_vending)}")
for (estado, evento), destino in delta_vending.items():
    print(f"{estado:10} -- {evento:15} --> {destino}")


def processar_venda(eventos, mostrar=True):
    estado = "C0"
    credito = Decimal("0.00")
    if mostrar:
        print(f"\n🛒 Fluxo: {eventos}")
    for passo, evento in enumerate(eventos, start=1):
        destino = delta_vending.get((estado, evento))
        if destino is None:
            permitidos = [ev for (q, ev) in delta_vending if q == estado]
            if mostrar:
                print(f"❌ Evento {passo} {evento!r} recusado em {estado}.")
                print(f"   Eventos permitidos: {permitidos or 'nenhum; a venda já terminou'}")
            return False

        estado_anterior = estado
        estado = destino
        if evento.startswith("INSERIR "):
            credito += Decimal(evento.split()[1])
            if mostrar:
                print(f"{estado_anterior:10} -- {evento:15} --> {estado:10} | crédito = R$ {credito:.2f}")
        elif evento == "ESCOLHER B3":
            if mostrar:
                print(f"{estado_anterior:10} -- {evento:15} --> {estado:10} | B3 custa R$ {PRECO_B3:.2f}")
        elif evento == "TROCO":
            troco = credito - PRECO_B3
            if mostrar:
                print(f"{estado_anterior:10} -- {evento:15} --> {estado:10} | troco = R$ {troco:.2f}")

    if estado in ESTADOS_FINAIS_VENDING:
        if mostrar:
            print("✅ Venda concluída.")
        return True
    if mostrar:
        print(f"⏳ Fluxo incompleto: terminou em {estado}, não em um estado final {ESTADOS_FINAIS_VENDING}.")
    return False

print("\n--- Dois fluxos válidos ---")
processar_venda(["INSERIR 1.00", "INSERIR 0.50", "ESCOLHER B3", "TROCO"])
processar_venda(["INSERIR 0.50", "INSERIR 0.50", "INSERIR 0.25", "ESCOLHER B3", "TROCO"])

print("\n--- Dois fluxos inválidos ---")
processar_venda(["ESCOLHER B3"])  # crédito insuficiente; lista moedas permitidas
processar_venda(["INSERIR 1.00", "INSERIR 0.50", "INSERIR 0.25"])  # ultrapassa limite de 1.50

Tabela de transições da vending machine:
Total de estados: 9
Total de transições: 17
C0         -- INSERIR 0.25    --> C025
C0         -- INSERIR 0.50    --> C050
C0         -- INSERIR 1.00    --> C100
C025       -- INSERIR 0.25    --> C050
C025       -- INSERIR 0.50    --> C075
C025       -- INSERIR 1.00    --> C125
C050       -- INSERIR 0.25    --> C075
C050       -- INSERIR 0.50    --> C100
C050       -- INSERIR 1.00    --> C150
C075       -- INSERIR 0.25    --> C100
C075       -- INSERIR 0.50    --> C125
C100       -- INSERIR 0.25    --> C125
C100       -- INSERIR 0.50    --> C150
C125       -- INSERIR 0.25    --> C150
C125       -- ESCOLHER B3     --> LIBERANDO
C150       -- ESCOLHER B3     --> LIBERANDO
LIBERANDO  -- TROCO           --> TROCO

--- Dois fluxos válidos ---

🛒 Fluxo: ['INSERIR 1.00', 'INSERIR 0.50', 'ESCOLHER B3', 'TROCO']
C0         -- INSERIR 1.00    --> C100       | crédito = R$ 1.00
C100       -- INSERIR 0.50    --> C150       | crédito = R$ 1.50
C150       -- E

False

## R5 — Mini-scanner da linguagem de comandos

Comandos reconhecidos: `INSERIR`, `ESCOLHER` e `TROCO`. O scanner ignora espaços, tabulações, quebras de linha e comentários `#`. Ele informa linha, coluna e dica para caracteres ou moedas inválidos e **continua a varredura** depois do erro.

O Maximal Munch é demonstrado nos operadores: o scanner tenta `==`, `<=`, `>=` e `!=` antes de `=`, `<` e `>`. Assim, `==` vira um único token, não dois tokens `=`.

In [17]:
# R5 — Mini-scanner específico da vending machine; inspirado no escanear() da aula
PALAVRAS_RESERVADAS_VENDING = {"INSERIR", "ESCOLHER", "TROCO"}
REGEX_FORMATO_MOEDA_VENDING = re.compile(REGEX_MOEDA)
REGEX_FORMATO_CANDIDATO_PRODUTO = re.compile(r"[A-Za-z][0-9]")
OPERADORES_VENDING = [
    ("==", "OP_COMP"), ("<=", "OP_COMP"), (">=", "OP_COMP"), ("!=", "OP_COMP"),
    ("=", "ATRIB"), ("<", "OP_COMP"), (">", "OP_COMP"),
]
SIMPLES_VENDING = {
    "(": "PAREN_ESQ", ")": "PAREN_DIR", ";": "PONTO_VIRGULA",
    "+": "MAIS", "-": "MENOS",
}


def escanear_vending(fonte):
    tokens, erros = [], []
    i, linha, coluna = 0, 1, 1

    def registrar_erro(lin, col, mensagem):
        erros.append(f"Linha {lin}, coluna {col}: {mensagem}")

    while i < len(fonte):
        c = fonte[i]
        inicio, linha_inicio, col_inicio = i, linha, coluna

        if c == "\n":
            i += 1
            linha, coluna = linha + 1, 1
            continue
        if c in " \t\r":
            i += 1
            coluna += 1
            continue
        if c == "#":
            while i < len(fonte) and fonte[i] != "\n":
                i += 1
            coluna += i - inicio
            continue

        # Maximal Munch: testa os operadores de dois caracteres antes dos de um.
        operador_encontrado = next(
            ((op, tipo) for op, tipo in OPERADORES_VENDING if fonte.startswith(op, i)),
            None,
        )
        if operador_encontrado:
            lexema, tipo = operador_encontrado
            i += len(lexema)
            tokens.append((tipo, lexema, linha_inicio, col_inicio))
            coluna += len(lexema)
            continue

        if c.isalpha() or c == "_":
            i += 1
            while i < len(fonte) and (fonte[i].isalnum() or fonte[i] == "_"):
                i += 1
            lexema = fonte[inicio:i]
            upper = lexema.upper()
            if upper in PALAVRAS_RESERVADAS_VENDING:
                tipo = upper
            elif REGEX_FORMATO_CANDIDATO_PRODUTO.fullmatch(lexema):
                # A forma é lexicalmente produto; o AFD separadamente verifica A-F e 1-9.
                tipo = "COD_PRODUTO"
            else:
                tipo = "ID"
            tokens.append((tipo, lexema, linha_inicio, col_inicio))
            coluna += i - inicio
            continue

        if c.isdigit():
            i += 1
            while i < len(fonte) and fonte[i].isdigit():
                i += 1
            if i + 1 < len(fonte) and fonte[i] == "." and fonte[i + 1].isdigit():
                i += 1
                while i < len(fonte) and fonte[i].isdigit():
                    i += 1
                lexema = fonte[inicio:i]
                if REGEX_FORMATO_MOEDA_VENDING.fullmatch(lexema):
                    tipo = "MOEDA"
                else:
                    tipo = "NUM_REAL"
                    registrar_erro(
                        linha_inicio, col_inicio,
                        f"moeda {lexema!r} não aceita; use 0.25, 0.50 ou 1.00 (zeros à esquerda são permitidos).",
                    )
            else:
                lexema = fonte[inicio:i]
                tipo = "NUM_INT"
            tokens.append((tipo, lexema, linha_inicio, col_inicio))
            coluna += i - inicio
            continue

        if c in SIMPLES_VENDING:
            i += 1
            tokens.append((SIMPLES_VENDING[c], c, linha_inicio, col_inicio))
            coluna += 1
            continue

        # Dicas específicas do domínio; o scanner avança um caractere e continua.
        if c == ",":
            dica = "use ponto decimal para moedas, por exemplo 0.50."
        elif c == "$":
            dica = "não inclua o símbolo R$; informe apenas 0.25, 0.50 ou 1.00."
        elif c == "@":
            dica = "comando desconhecido; use INSERIR, ESCOLHER ou TROCO."
        elif c == ".":
            dica = "o ponto só é aceito dentro de um valor decimal, como 1.00."
        else:
            dica = "caractere não reconhecido pela linguagem da vending machine."
        registrar_erro(linha_inicio, col_inicio, f"caractere inesperado {c!r}; {dica}")
        i += 1
        coluna += 1

    return tokens, erros


programa_vending = """# Compra de café
INSERIR 1.00
INSERIR 0.50
ESCOLHER B3  # café grande
TROCO
"""
tokens_vending, erros_vending = escanear_vending(programa_vending)
print(f"{'TOKEN':18} {'LEXEMA':14} {'LIN':>3} {'COL':>3}")
for tipo, lexema, lin, col in tokens_vending:
    print(f"{tipo:18} {lexema:14} {lin:>3} {col:>3}")
print("Erros:", erros_vending or "nenhum")

print("\n--- Maximal Munch: == vs =, <= vs <, moeda vs inteiro ---")
amostra_maximal = "A1 == B3; A1 = B3; A1 <= F9; A1 < F9; 1.00 1"
toks_mm, errs_mm = escanear_vending(amostra_maximal)
for tok in toks_mm:
    print(tok)
print("Erros:", errs_mm or "nenhum")

print("\n--- Continuação após erros léxicos ---")
fonte_com_erros = "INSERIR 0,50 @\nESCOLHER B3\nTROCO"
toks, errs = escanear_vending(fonte_com_erros)
for tok in toks:
    print(tok)
for erro in errs:
    print("⚠️", erro)

TOKEN              LEXEMA         LIN COL
INSERIR            INSERIR          2   1
MOEDA              1.00             2   9
INSERIR            INSERIR          3   1
MOEDA              0.50             3   9
ESCOLHER           ESCOLHER         4   1
COD_PRODUTO        B3               4  10
TROCO              TROCO            5   1
Erros: nenhum

--- Maximal Munch: == vs =, <= vs <, moeda vs inteiro ---
('COD_PRODUTO', 'A1', 1, 1)
('OP_COMP', '==', 1, 4)
('COD_PRODUTO', 'B3', 1, 7)
('PONTO_VIRGULA', ';', 1, 9)
('COD_PRODUTO', 'A1', 1, 11)
('ATRIB', '=', 1, 14)
('COD_PRODUTO', 'B3', 1, 16)
('PONTO_VIRGULA', ';', 1, 18)
('COD_PRODUTO', 'A1', 1, 20)
('OP_COMP', '<=', 1, 23)
('COD_PRODUTO', 'F9', 1, 26)
('PONTO_VIRGULA', ';', 1, 28)
('COD_PRODUTO', 'A1', 1, 30)
('OP_COMP', '<', 1, 33)
('COD_PRODUTO', 'F9', 1, 35)
('PONTO_VIRGULA', ';', 1, 37)
('MOEDA', '1.00', 1, 39)
('NUM_INT', '1', 1, 44)
Erros: nenhum

--- Continuação após erros léxicos ---
('INSERIR', 'INSERIR', 1, 1)
('NUM_INT', '0'

## R6 — Painel interativo

O painel permite escolher um dos dois validadores ou testar o mini-scanner. Digite um código, uma moeda ou um programa de comandos e clique em **Testar**.

In [18]:
# R6 — Interface interativa com ipywidgets
import ipywidgets as widgets
from IPython.display import display, clear_output

modo_vending = widgets.Dropdown(
    options=[("Validador: código do produto", "produto"),
             ("Validador: moeda", "moeda"),
             ("Mini-scanner", "scanner")],
    value="produto",
    description="Módulo:",
    layout=widgets.Layout(width="390px"),
)
entrada_vending = widgets.Textarea(
    value="B3",
    description="Entrada:",
    layout=widgets.Layout(width="90%", height="100px"),
)
botao_vending = widgets.Button(description="Testar", button_style="primary")
saida_vending = widgets.Output()


def testar_vending(_):
    saida_vending.clear_output()
    with saida_vending:
        if modo_vending.value == "produto":
            resultado = afd_codigo_produto.rodar(entrada_vending.value)
            regex_ok = re.fullmatch(REGEX_CODIGO_PRODUTO, entrada_vending.value) is not None
            print(f"RegEx equivalente: {regex_ok}; AFD: {resultado}")
        elif modo_vending.value == "moeda":
            resultado = afd_moeda.rodar(entrada_vending.value)
            regex_ok = re.fullmatch(REGEX_MOEDA, entrada_vending.value) is not None
            print(f"RegEx equivalente: {regex_ok}; AFD: {resultado}")
        else:
            toks, errs = escanear_vending(entrada_vending.value)
            print(f"{'TOKEN':18} {'LEXEMA':14} {'LIN':>3} {'COL':>3}")
            for tipo, lexema, lin, col in toks:
                print(f"{tipo:18} {lexema:14} {lin:>3} {col:>3}")
            print("\nErros:")
            print("\n".join(errs) if errs else "Nenhum")

botao_vending.on_click(testar_vending)
display(widgets.VBox([modo_vending, entrada_vending, botao_vending, saida_vending]))

## R7 — Resumo para o README

- **Tema:** autômatos e scanner de comandos para máquina de café / vending machine.
- **Validadores:** código de produto (`A1`–`F9`) e moedas aceitas (`0.25`, `0.50`, `1.00`, com zeros à esquerda opcionais).
- **Máquina de estados:** acumula crédito até R$ 1,50, libera o produto `B3` a R$ 1,25 e calcula o troco.
- **Scanner:** reconhece comandos, moedas, códigos, operadores e pontuação; ignora espaços e comentários; reporta erros com linha, coluna e dica e continua a leitura.
- **Maximal Munch:** `==` é reconhecido antes de `=`, assim como `<=`, `>=` e `!=` antes dos operadores menores.
- **Execução:** rode as células originais e, em seguida, as novas células R1–R6 na ordem. O painel interativo requer `ipywidgets`.

**Integrantes:** substitua esta linha pelos nomes de todos os integrantes do grupo antes de publicar o repositório.